In [2]:
import findspark
findspark.init()

from pyspark.conf import SparkConf
from pyspark.sql import SparkSession
import pyspark.sql.functions as F

conf = SparkConf().setAppName("1341").setMaster("local[4]")
spark = SparkSession.builder.config(conf= conf).getOrCreate()
spark

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
25/08/11 01:10:48 WARN Utils: Your hostname, de24, resolves to a loopback address: 127.0.1.1; using 192.168.0.102 instead (on interface enp0s3)
25/08/11 01:10:48 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/08/11 01:10:50 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [ ]:
'''
Table: Movies

+---------------+---------+
| Column Name   | Type    |
+---------------+---------+
| movie_id      | int     |
| title         | varchar |
+---------------+---------+
movie_id is the primary key (column with unique values) for this table.
title is the name of the movie.
 

Table: Users

+---------------+---------+
| Column Name   | Type    |
+---------------+---------+
| user_id       | int     |
| name          | varchar |
+---------------+---------+
user_id is the primary key (column with unique values) for this table.
 

Table: MovieRating

+---------------+---------+
| Column Name   | Type    |
+---------------+---------+
| movie_id      | int     |
| user_id       | int     |
| rating        | int     |
| created_at    | date    |
+---------------+---------+
(movie_id, user_id) is the primary key (column with unique values) for this table.
This table contains the rating of a movie by a user in their review.
created_at is the user's review date. 
 

Write a solution to:

Find the name of the user who has rated the greatest number of movies. 
In case of a tie, return the lexicographically smaller user name.
Find the movie name with the highest average rating in February 2020. 
In case of a tie, return the lexicographically smaller movie name.
The result format is in the following example.

 

Example 1:

Input: 
Movies table:
+-------------+--------------+
| movie_id    |  title       |
+-------------+--------------+
| 1           | Avengers     |
| 2           | Frozen 2     |
| 3           | Joker        |
+-------------+--------------+
Users table:
+-------------+--------------+
| user_id     |  name        |
+-------------+--------------+
| 1           | Daniel       |
| 2           | Monica       |
| 3           | Maria        |
| 4           | James        |
+-------------+--------------+
MovieRating table:
+-------------+--------------+--------------+-------------+
| movie_id    | user_id      | rating       | created_at  |
+-------------+--------------+--------------+-------------+
| 1           | 1            | 3            | 2020-01-12  |
| 1           | 2            | 4            | 2020-02-11  |
| 1           | 3            | 2            | 2020-02-12  |
| 1           | 4            | 1            | 2020-01-01  |
| 2           | 1            | 5            | 2020-02-17  | 
| 2           | 2            | 2            | 2020-02-01  | 
| 2           | 3            | 2            | 2020-03-01  |
| 3           | 1            | 3            | 2020-02-22  | 
| 3           | 2            | 4            | 2020-02-25  | 
+-------------+--------------+--------------+-------------+
Output: 
+--------------+
| results      |
+--------------+
| Daniel       |
| Frozen 2     |
+--------------+
Explanation: 
Daniel and Monica have rated 3 movies ("Avengers", "Frozen 2" and "Joker") 
but Daniel is smaller lexicographically.
Frozen 2 and Joker have a rating average of 3.5 in February 
but Frozen 2 is smaller lexicographically.
'''

In [3]:
movies_data = [
(1,'Avengers'),
(2,'Frozen 2'),
(3,'Joker')
]
movies_schema = ['movie_id','title']
users_data = [
(1,'Daniel'),
(2,'Monica'),
(3,'Maria'),
(4,'James')
]
users_schema = ['user_id','name']
moviesrating_data = [
(1,1,3,'2020-01-12'),
(1,2,4,'2020-02-11'),
(1,3,2,'2020-02-12'),
(1,4,1,'2020-01-01'),
(2,1,5,'2020-02-17'), 
(2,2,2,'2020-02-01'), 
(2,3,2,'2020-03-01'),
(3,1,3,'2020-02-22'), 
(3,2,4,'2020-02-25')
]
moviesrating_schema = ['movie_id','user_id','rating','created_at']

In [4]:
movies_df = spark.createDataFrame(data = movies_data, schema = movies_schema)
users_df = spark.createDataFrame(data = users_data, schema = users_schema)
moviesrating_df = spark.createDataFrame(data = moviesrating_data, schema = moviesrating_schema)

In [5]:
movies_df.show()
users_df.show()
moviesrating_df.show()

+--------+--------+
|movie_id|   title|
+--------+--------+
|       1|Avengers|
|       2|Frozen 2|
|       3|   Joker|
+--------+--------+

+-------+------+
|user_id|  name|
+-------+------+
|      1|Daniel|
|      2|Monica|
|      3| Maria|
|      4| James|
+-------+------+

+--------+-------+------+----------+
|movie_id|user_id|rating|created_at|
+--------+-------+------+----------+
|       1|      1|     3|2020-01-12|
|       1|      2|     4|2020-02-11|
|       1|      3|     2|2020-02-12|
|       1|      4|     1|2020-01-01|
|       2|      1|     5|2020-02-17|
|       2|      2|     2|2020-02-01|
|       2|      3|     2|2020-03-01|
|       3|      1|     3|2020-02-22|
|       3|      2|     4|2020-02-25|
+--------+-------+------+----------+



In [29]:
temp_df = moviesrating_df.alias("mr").join( users_df.alias("u"), 
                                              F.col("mr.user_id") == F.col("u.user_id"),
                                              "left"
                                            )\
                                     .groupBy(F.col("u.name"))\
                                     .agg(  F.count(F.col("*")).alias("counts") ,
                                            F.length(F.col("u.name")).alias("lengths")
                                         )\
                                     .orderBy(F.col("counts").desc(),F.col("u.name"), F.col("lengths").asc())\
                                     .select(F.col("u.name").alias("results"))\
                                     .limit(1)
temp_df.show()

+-------+
|results|
+-------+
| Daniel|
+-------+



In [42]:
temp1_df = moviesrating_df.alias("mr").join( movies_df.alias("m"), 
                                              F.col("mr.movie_id") == F.col("m.movie_id"),
                                              "left"
                                            )\
                                     .where((F.year(F.col("created_at")) == 2020) & (F.month(F.col("created_at")) == 2))\
                                     .groupBy(F.col("m.title"))\
                                     .agg(  F.avg(F.col("mr.rating")).alias("ratings") ,
                                            F.length(F.col("m.title")).alias("lengths")
                                         )\
                                     .orderBy(F.col("ratings").desc(),F.col("m.title").asc(), F.col("lengths").asc())\
                                     .select(F.col("m.title").alias("results"))\
                                     .limit(1)
temp1_df.show()

+--------+
| results|
+--------+
|Frozen 2|
+--------+



In [43]:
temp_df = moviesrating_df.alias("mr").join( users_df.alias("u"), 
                                              F.col("mr.user_id") == F.col("u.user_id"),
                                              "left"
                                            )\
                                     .groupBy(F.col("u.name"))\
                                     .agg(  F.count(F.col("*")).alias("counts") ,
                                            F.length(F.col("u.name")).alias("lengths")
                                         )\
                                     .orderBy(F.col("counts").desc(),F.col("u.name"), F.col("lengths").asc())\
                                     .select(F.col("u.name").alias("results"))\
                                     .limit(1)
temp1_df = moviesrating_df.alias("mr").join( movies_df.alias("m"), 
                                              F.col("mr.movie_id") == F.col("m.movie_id"),
                                              "left"
                                            )\
                                     .where((F.year(F.col("created_at")) == 2020) & (F.month(F.col("created_at")) == 2))\
                                     .groupBy(F.col("m.title"))\
                                     .agg(  F.avg(F.col("mr.rating")).alias("ratings") ,
                                            F.length(F.col("m.title")).alias("lengths")
                                         )\
                                     .orderBy(F.col("ratings").desc(),F.col("m.title").asc(), F.col("lengths").asc())\
                                     .select(F.col("m.title").alias("results"))\
                                     .limit(1)

temp_df.union(temp1_df).show()

+--------+
| results|
+--------+
|  Daniel|
|Frozen 2|
+--------+



## SQL Solution
<pre>
WITH TEMP AS (
    SELECT u.name , count(*) as counts, LENGTH(u.name) as lengths
    FROM MovieRating mr 
        LEFT JOIN Users ON u.user_id = mr.user_id
        GROUP BY u.name
        ORDER BY counts desc, lengths limit 1
)
SELECT name as results
FROM TEMP

UNION

WITH TEMP1 AS (
    SELECT m.title, count(*) as counts, LENGTH(m.title) as lengths
    FROM MovieRating mr
        LEFT JOIN Movies m ON mr.movie_id = m.movie_id
        WHERE YEAR(created_at) = 2020 and MONTH(created_at) = 2
        GROUP BY m.title
        ORDER BY counts desc, m.title lengths limit 1
)
SELECT title as results
FROM TEMP

</pre>